# Apple Detector — Kaggle Training

Fine-tunes Faster R-CNN (ResNet-50 FPN) on a COCO-format apple detection dataset.
Expected Kaggle input dataset layout:
```
<DATASET_ROOT>/
  annotations/
    instances_train.json
    instances_val.json
  images/
    train/   *.png / *.jpg
    val/     *.png / *.jpg
```

Upload a merged COCO dataset (e.g. from `scripts/merge_coco_datasets.py`) to Kaggle
and update `DATASET_ROOT` below to match its path.

## Setup

In [ ]:
!git clone https://github.com/philippsnr/apple-vision.git
%cd apple-vision

In [ ]:
!cd /kaggle/working/apple-vision && git pull && uv sync --quiet

## Dataset Check

In [ ]:
import os, json

DATASET_ROOT = '/kaggle/input/datasets/philippstaudinger/project-anna/apple_mots_coco/apple_mots_coco'

for split in ['train', 'val']:
    ann = f'{DATASET_ROOT}/annotations/instances_{split}.json'
    img_dir = f'{DATASET_ROOT}/images/{split}'
    if os.path.exists(ann):
        data = json.load(open(ann))
        n_img = len(data.get('images', []))
        n_ann = len(data.get('annotations', []))
        print(f"{split}: {n_img} images, {n_ann} annotations")
    else:
        print(f"{split}: MISSING — check DATASET_ROOT")

## Train

In [ ]:
!cd /kaggle/working/apple-vision && MPLBACKEND=agg uv run python -m apple_vision.train \
  --dataset-root /kaggle/input/datasets/philippstaudinger/project-anna/apple_mots_coco/apple_mots_coco \
  --epochs 30 \
  --batch-size 4 \
  --num-workers 4 \
  --early-stop-patience 5 \
  --out-dir /kaggle/working/checkpoints

## Evaluate — COCO mAP

In [ ]:
!cd /kaggle/working/apple-vision && uv run python -m apple_vision.evaluate_coco \
  --dataset-root /kaggle/input/datasets/philippstaudinger/project-anna/apple_mots_coco/apple_mots_coco \
  --checkpoint /kaggle/working/checkpoints/fasterrcnn_resnet50_fpn_apple_best.pth \
  --results-json /kaggle/working/checkpoints/coco_results.json

## Visualize — Predictions vs Ground Truth

In [ ]:
!cd /kaggle/working/apple-vision && MPLBACKEND=agg uv run python -m apple_vision.visualize_detections \
  --checkpoint /kaggle/working/checkpoints/fasterrcnn_resnet50_fpn_apple_best.pth \
  --dataset-root /kaggle/input/datasets/philippstaudinger/project-anna/apple_mots_coco/apple_mots_coco \
  --score-threshold 0.5 \
  --n 8 \
  --out-dir /kaggle/working/quickplots/detections

In [ ]:
from IPython.display import display, Image
from pathlib import Path

for p in sorted(Path('/kaggle/working/quickplots/detections').glob('*.png'))[:4]:
    display(Image(filename=str(p)))